In [ ]:
# Mount Google Drive to access input data
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [ ]:
# Install required libraries
# xarray / netCDF4 : NetCDF dataset I/O
# rioxarray        : raster I/O and spatial operations
# scipy            : not used in Figure 6 but kept for downstream compatibility
!pip install xarray netCDF4 scipy pandas matplotlib rioxarray


In [ ]:
# ============================================================
# Imports and Configuration
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
import xarray as xr
import pandas as pd
import rioxarray
import glob
import re

# --- File paths ---
# NOAA CRW daily SST and DHW NetCDF (downloaded from CoralTemp data portal)
CRW_PATH          = "/content/drive/MyDrive/<YOUR_PROJECT_FOLDER>/NOAA CRW/Data/<YOUR_CRW_FILE>.nc"

# NOAA CRW climatology NetCDF (ct5km_climatology_v3.1)
CLIM_PATH         = "/content/drive/MyDrive/<YOUR_PROJECT_FOLDER>/NOAA CRW/Data/ct5km_climatology_v3.1.nc"

# In-situ bleaching survey CSV with YearMonth and Bleached_Percent columns
BLEACHING_CSV     = "/content/drive/MyDrive/<YOUR_PROJECT_FOLDER>/NOAA CRW/Data/bleaching_confusion_labeled.csv"

# Folder containing monthly ECOSTRESS LST median composites
# Expected filename pattern: masked_ECOSTRESS_LST_Median_YYYY-MM*.tif
ECOSTRESS_FOLDER  = "/content/drive/MyDrive/<YOUR_PROJECT_FOLDER>/ECOSTRESS Tifs/Geolocated LST/Composites/new_landmasked_output"

# --- Study area bounds (Belize Barrier Reef System) ---
LAT_MIN, LAT_MAX =  15.7,  18.5
LON_MIN, LON_MAX = -89.5, -87.3

# --- Time window ---
TIME_START = "2023-05-01"
TIME_END   = "2024-04-30"

# --- DHW alert color scheme (NOAA CRW standard) ---
COLOR_WATCH   = '#FDEE66'  # Yellow  — Watch/Warning (0 < DHW < 4)
COLOR_WARNING = '#F2A03D'  # Orange  — Alert Level 1 (4 ≤ DHW < 8)
COLOR_ALERT1  = '#D73B28'  # Red     — Alert Level 2 (8 ≤ DHW < 12)
COLOR_ALERT2  = '#9E0028'  # Dark red — Alert Level 3 (DHW ≥ 12)
COLOR_ALERT3  = '#5C001A'  # Very dark red — Alert Level 4 (16 ≤ DHW < 20)


In [ ]:
# ============================================================
# Figure 6: Regional Thermal Stress and Coral Bleaching
#           Time Series — Belize, May 2023 – April 2024
# ============================================================
# Dual-axis time series figure showing:
#
# Left y-axis (blue) — Temperature (°C):
#   - CRW climatological mean SST (1985–1993 baseline, dashed gray)
#   - CRW observed monthly mean SST (solid blue)
#   - ECOSTRESS monthly median LST (dotted green, triangles at
#     actual observations, linear interpolation for gaps)
#   - Regional MMM bleaching threshold (dashed blue horizontal line)
#   - DHW-based heat stress shading above the MMM threshold:
#       Yellow  = Watch/Warning  (0 < DHW < 4)
#       Orange  = Alert Level 1  (4 ≤ DHW < 8)
#       Red     = Alert Level 2  (8 ≤ DHW < 12)
#       Dark red = Alert Level 3 (DHW ≥ 12)
#       Deepest red = Alert Level 4 (16 ≤ DHW < 20)
#
# Right y-axis (red) — Bleached Coral (%):
#   - Monthly bleaching distribution as boxplots (no outliers)
#   - Survey count (n=) annotated above each box
#
# Note: The June 2023 ECOSTRESS file is excluded as it contains
# known data quality issues.
# ============================================================

# --- Load datasets ---
ds_anom = xr.open_dataset(CRW_PATH)
ds_clim = xr.open_dataset(CLIM_PATH)

# Adjust climatology longitudes if the dataset uses 0–360 convention
def adjust_longitudes(ds, lon_min, lon_max):
    """
    Convert negative (W) longitude bounds to 0–360 if the dataset
    stores longitudes in that convention.
    """
    if ds.lon.max() > 180:
        lon_min = lon_min + 360 if lon_min < 0 else lon_min
        lon_max = lon_max + 360 if lon_max < 0 else lon_max
    return lon_min, lon_max

lon_min_adj, lon_max_adj = adjust_longitudes(ds_clim, LON_MIN, LON_MAX)

# --- Climatology: regional MMM baseline and monthly means ---
# MMM (Maximum Monthly Mean) is the NOAA CRW bleaching onset threshold
bleaching_threshold = float(
    ds_clim['sst_clim_mmm'].sel(
        lat=slice(LAT_MAX, LAT_MIN),
        lon=slice(lon_min_adj, lon_max_adj)
    ).mean().values
)

month_names = ["january", "february", "march", "april", "may", "june",
               "july", "august", "september", "october", "november", "december"]
monthly_means = {}
for month in month_names:
    varname = f"sst_clim_{month}"
    if varname in ds_clim.variables:
        monthly_means[month] = float(
            ds_clim[varname].sel(
                lat=slice(LAT_MAX, LAT_MIN),
                lon=slice(lon_min_adj, lon_max_adj)
            ).mean().values
        )

clim_df = pd.DataFrame({
    "Month"       : [m.capitalize() for m in month_names],
    "Climatology" : [monthly_means[m] for m in month_names]
})
clim_df["MonthNum"] = pd.to_datetime(clim_df["Month"], format="%B").dt.month
clim_df.set_index("MonthNum", inplace=True)

# --- Observed CRW SST: daily series (for shading) and monthly mean (for line) ---
sst = ds_anom['CRW_SST'].sel(
    time=slice(TIME_START, TIME_END),
    latitude=slice(LAT_MAX, LAT_MIN),
    longitude=slice(LON_MIN, LON_MAX)
)
sst_series          = sst.mean(dim=["latitude", "longitude"]).to_series()  # daily
sst_monthly_mean_ts = sst_series.resample('MS').mean()                     # monthly

aligned_clim_ts = pd.Series(
    sst_monthly_mean_ts.index.month.map(clim_df['Climatology']),
    index=sst_monthly_mean_ts.index
)

# --- DHW: daily series (for heat stress shading) ---
dhw_series = ds_anom['CRW_DHW'].sel(
    time=slice(TIME_START, TIME_END),
    latitude=slice(LAT_MAX, LAT_MIN),
    longitude=slice(LON_MIN, LON_MAX)
).mean(dim=["latitude", "longitude"]).to_series()

# --- In-situ bleaching: monthly distributions for boxplots ---
df_bleach = pd.read_csv(BLEACHING_CSV)
df_bleach['time'] = pd.to_datetime(df_bleach['YearMonth'], format='%Y-%m')
bleach_monthly_data   = df_bleach.groupby('time')['Bleached_Percent'].apply(list)
bleach_survey_counts  = df_bleach.groupby('time').size()
common_index          = sst_monthly_mean_ts.index.intersection(bleach_monthly_data.index)
bleach_monthly_data   = bleach_monthly_data.reindex(common_index)
bleach_survey_counts  = bleach_survey_counts.reindex(common_index)

# --- ECOSTRESS monthly median LST composites ---
# Files follow the pattern: masked_ECOSTRESS_LST_Median_YYYY-MM*.tif
# June 2023 is excluded due to known data quality issues.
ecostress_timestamps = []
ecostress_values     = []

for tif_path in sorted(glob.glob(f"{ECOSTRESS_FOLDER}/masked_ECOSTRESS_LST_Median_*.tif")):
    if "2023-06" in tif_path:  # skip known bad file
        continue
    match = re.search(r'(\d{4}-\d{2})', tif_path)
    if not match:
        continue
    timestamp = pd.to_datetime(match.group(1))
    raster    = rioxarray.open_rasterio(tif_path).rio.reproject("EPSG:4326")
    clipped   = raster.rio.clip_box(minx=LON_MIN, miny=LAT_MIN,
                                     maxx=LON_MAX, maxy=LAT_MAX, crs="EPSG:4326")
    ecostress_timestamps.append(timestamp)
    ecostress_values.append(clipped.where(clipped > 0).mean().item())

ecostress_ts             = pd.Series(ecostress_values, index=ecostress_timestamps)
ecostress_ts             = ecostress_ts.reindex(sst_monthly_mean_ts.index)
ecostress_ts_interpolated = ecostress_ts.interpolate(method='linear')  # fills gaps for line continuity

# ============================================================
# Plot
# ============================================================

fig, ax1 = plt.subplots(figsize=(16, 9))

# --- Heat stress shading (stacked, based on daily DHW) ---
# Each level overwrites the previous, producing nested colour bands
for where_cond, color in [
    ((dhw_series > 0)  & (dhw_series < 4)  & (sst_series > bleaching_threshold), COLOR_WATCH),
    ((dhw_series >= 4) & (dhw_series < 8)  & (sst_series > bleaching_threshold), COLOR_WARNING),
    ((dhw_series >= 8) & (dhw_series < 12) & (sst_series > bleaching_threshold), COLOR_ALERT1),
    ((dhw_series >= 12)                    & (sst_series > bleaching_threshold), COLOR_ALERT2),
    ((dhw_series >= 16) & (dhw_series < 20) & (sst_series > bleaching_threshold), COLOR_ALERT3),
]:
    ax1.fill_between(dhw_series.index, bleaching_threshold, sst_series,
                     where=where_cond, color=color, alpha=0.7,
                     zorder=[1,2,3,4,5][[COLOR_WATCH,COLOR_WARNING,COLOR_ALERT1,COLOR_ALERT2,COLOR_ALERT3].index(color)])

# --- Temperature lines (left axis) ---
color_temp = 'tab:blue'
ax1.set_xlabel("Month", fontsize=12)
ax1.set_ylabel("Sea Surface Temperature (\u00b0C)", color=color_temp, fontsize=14)
ax1.set_ylim(25, 50)

ln_threshold = ax1.axhline(y=bleaching_threshold, color='dodgerblue', linestyle='--',
                            linewidth=2, label='Regional MMM Baseline', zorder=5)
ln1 = ax1.plot(aligned_clim_ts.index, aligned_clim_ts.values,
               label="Climatological Mean SST", marker='o', linestyle='--',
               color='gray', linewidth=2, markersize=6, zorder=5)
ln2 = ax1.plot(sst_monthly_mean_ts.index, sst_monthly_mean_ts.values,
               label="Observed Mean SST (CRW)", marker='o',
               color=color_temp, linewidth=2, markersize=6, zorder=5)
ln3 = ax1.plot(ecostress_ts_interpolated.index, ecostress_ts_interpolated.values,
               label="Median LST (ECOSTRESS)", linestyle=':', color='green',
               linewidth=2, zorder=5)
ax1.plot(ecostress_ts.index, ecostress_ts.values,
         marker='^', linestyle='none', color='green', markersize=8, zorder=6)

ax1.tick_params(axis='y', labelcolor=color_temp, labelsize=12)
ax1.grid(True, alpha=0.3, linestyle=':', linewidth=0.5)

# --- Bleaching boxplots (right axis) ---
ax2 = ax1.twinx()
color_bleach = 'tab:red'
ax2.set_ylabel("Bleached Coral (%)", color=color_bleach, fontsize=14)
ax2.tick_params(axis='y', labelcolor=color_bleach, labelsize=12)
ax2.set_ylim(-15, 105)

box_positions = [mdates.date2num(d) for d in bleach_monthly_data.index]
bp = ax2.boxplot(bleach_monthly_data.values, positions=box_positions,
                 widths=15, patch_artist=True, showfliers=False)

for patch   in bp['boxes']:    patch.set_facecolor(color_bleach); patch.set_alpha(0.5); patch.set_edgecolor('darkred'); patch.set_linewidth(1.5)
for median  in bp['medians']:  median.set_color('black'); median.set_linewidth(2)
for whisker in bp['whiskers']: whisker.set_color('darkred'); whisker.set_linewidth(1.5)
for cap     in bp['caps']:     cap.set_color('darkred'); cap.set_linewidth(1.5)

# Survey count annotations above each box
for i, date in enumerate(bleach_monthly_data.index):
    ax2.text(mdates.date2num(date), max(bleach_monthly_data.iloc[i]) + 2,
             f"n={bleach_survey_counts.iloc[i]}",
             ha='center', va='bottom', fontsize=10, fontweight='bold', color='black')

# --- x-axis formatting ---
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax1.tick_params(axis='x', labelsize=12)
fig.autofmt_xdate(rotation=0, ha='center')

# --- Legend ---
legend_elements = [
    ln1[0],
    ln2[0],
    ln3[0],
    ln_threshold,
    Patch(facecolor=color_bleach, alpha=0.5, edgecolor='darkred', label='Bleaching Distribution (%)'),
    Patch(facecolor=COLOR_WATCH,   alpha=0.7, label='Watch/Warning (0 < DHW < 4)'),
    Patch(facecolor=COLOR_WARNING, alpha=0.7, label='Alert Level 1 (4 \u2264 DHW < 8)'),
    Patch(facecolor=COLOR_ALERT1,  alpha=0.7, label='Alert Level 2 (8 \u2264 DHW < 12)'),
    Patch(facecolor=COLOR_ALERT2,  alpha=0.7, label='Alert Level 3 (DHW \u2265 12)'),
    Patch(facecolor=COLOR_ALERT3,  alpha=0.7, label='Alert Level 4 (16 \u2264 DHW < 20)'),
]
ax1.legend(handles=legend_elements, loc='upper right', fontsize=12,
           framealpha=0.95, edgecolor='black')

ax1.set_title("Temperature and Coral Bleaching in Belize (2023\u20132024)",
              fontsize=18, fontweight='bold', pad=20)
fig.tight_layout()
plt.show()
